# STIR-Net V1 — 19 fast spatial instance-decomposition diagnostic

This notebook deliberately **stops temporal-routing development** for this scene and asks a more fundamental question:

> **Can the current-frame spatial pathway separate the short-contact / touching cells by itself?**

For this sample, temporal evidence should be optional assistance. If obvious touching cells cannot be decomposed from the current volume, the immediate bottleneck is in the spatial instance-decomposition pathway.

This notebook contains **no full overfit**.

## What is tested

### Gate A — pure-spatial dense evidence
Using the final learned weights but bypassing temporal co-reasoning:

- internal GT-boundary evidence inside source 9;
- dense center-heatmap evidence;
- how many spatial center peaks are visible inside source 9;
- peak-to-GT-center distances.

### Gate B — spatial feature separability
At E2 and D1:

- per-GT pooled feature similarity;
- nearest-centroid voxel classification inside the source-9 GT cells.

This is supplementary evidence about whether the spatial representation itself distinguishes the cells.

### Gate C — spatial query counterfactuals
Temporal memory is removed completely.

The same learned spatial features are decoded under:

1. **baseline** — all source-9 seeded queries start at their normal shared source centroid;
2. **oracle centers** — diagnostic only: source-9 queries are placed at the nine GT centers;
3. **spatial-peak centers** — queries are initialized from the learned dense center-heatmap peaks;
4. **oracle + split-local support** — GT centers plus removal of the full merged-source support for split queries;
5. **peak + split-local support** — legitimate spatial peaks plus local split support.

This localizes whether the bottleneck is:

- spatial representation,
- query/reference initialization,
- whole-component attention support,
- or later mask decoding.

### Gate D — 5-step cached spatial micro-overfit
The expensive CNN is run once and detached.

Two short branches train only QueryBuilder + QueryDecoder:

- normal shared source support;
- split-local source support.

All real scene queries and real Hungarian matching are retained.

No native rendering. No temporal branch. No full overfit.

## Interpretation principle

A failure of this notebook is **not automatically a CNN-encoder failure**. The spatial pathway includes:

```text
CNN features
→ component pooling
→ split-query construction
→ spatial support / cross-attention
→ center updates
→ coarse masks
```

The purpose is to find the first stage that cannot separate the easy touching cells.

In [ ]:
from pathlib import Path
from dataclasses import replace
from types import MethodType
import copy
import gc
import json
import math
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from scipy.optimize import linear_sum_assignment
from scipy.stats import rankdata

from learned.stirnet import RefinementCriterion, StirNet
from learned.stirnet.debugging.acceptance.first_overfit import (
    _reduced_config,
    _repo_root,
    build_real_batch,
)
from learned.stirnet.model.coordinates import (
    feature_grid_coordinates_um,
    resize_label_map_nearest,
)
from learned.stirnet.model.query_builder import (
    QUERY_PRIMARY,
    QUERY_SPLIT,
)
from learned.stirnet.model.types import TemporalState
from learned.stirnet.training.checkpoint import load_checkpoint
from learned.stirnet.training.curriculum import curriculum_stage
from learned.stirnet.training.trainer import move_to_device

SEED = 40266
SOURCE_ID = 9
AMP_DTYPE = torch.float16
MICRO_STEPS = 5
EVAL_STEPS = {0, 1, 3, 5}

REPO_ROOT = _repo_root(Path.cwd())
DATA_DIR = (
    REPO_ROOT
    / "data"
    / "learned"
    / "stirnet"
    / "first_overfit"
    / "BlastoSPIM1_F22_030_034"
)

NB15_RUN = (
    REPO_ROOT
    / "runs"
    / "stirnet"
    / "first_overfit"
    / "15_hierarchical_temporal_memory"
)
FINAL_CHECKPOINT = NB15_RUN / "checkpoint_joint.pt"

# Fallback for older local runs if the NB15 filename differs.
if not FINAL_CHECKPOINT.exists():
    candidates = [
        NB15_RUN / "step85_joint.pt",
        REPO_ROOT
        / "runs"
        / "stirnet"
        / "first_overfit"
        / "12_staged_same_sample"
        / "checkpoint_joint.pt",
    ]
    for candidate in candidates:
        if candidate.exists():
            FINAL_CHECKPOINT = candidate
            break

RUN_DIR = (
    REPO_ROOT
    / "runs"
    / "stirnet"
    / "targeted"
    / "19_spatial_decomposition"
)
RUN_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("Notebook 19 requires CUDA.")
if not FINAL_CHECKPOINT.exists():
    raise FileNotFoundError(
        "Could not find a trained joint checkpoint. Expected NB15/NB12 final checkpoint."
    )

device = torch.device("cuda")

print("Repository :", REPO_ROOT)
print("Data       :", DATA_DIR)
print("Checkpoint :", FINAL_CHECKPOINT)
print("Run dir    :", RUN_DIR)
print("GPU        :", torch.cuda.get_device_name(0))

## 1. Load the real scene and identify the GT cells contained in source component 9

This is derived directly from voxel overlap between the current CC label `9` and the GT instance map.

GT information is used only for diagnosis/evaluation.

In [ ]:
batch, sample = build_real_batch(DATA_DIR)
target = batch["targets"][0]

cfg = _reduced_config()
cfg.curriculum.enabled = True
cfg.curriculum.spatial_dense_steps = 30
cfg.curriculum.temporal_dense_steps = 20
cfg.curriculum.query_bootstrap_steps = 20
cfg.curriculum.native_bootstrap_steps = 10

def prepare_device_batch(cpu_batch):
    result = {}
    for key, value in cpu_batch.items():
        if key == "targets":
            result[key] = value
        elif key == "spatial_inputs":
            result[key] = value.to(
                device=device,
                dtype=AMP_DTYPE,
                non_blocking=True,
            )
        elif key == "instance_labels":
            result[key] = value.to(
                device=device,
                dtype=torch.int32,
                non_blocking=True,
            )
        else:
            result[key] = move_to_device(
                value,
                device,
            )
    return result

b = prepare_device_batch(batch)

current_labels_cpu = batch[
    "instance_labels"
][0].detach().cpu().numpy()

gt_label_map_cpu = torch.as_tensor(
    target["label_map"]
).detach().cpu().numpy()

source_mask_cpu = (
    current_labels_cpu == SOURCE_ID
)

overlap_gt_ids, overlap_counts = np.unique(
    gt_label_map_cpu[source_mask_cpu],
    return_counts=True,
)

positive = overlap_gt_ids > 0
source_gt_ids = overlap_gt_ids[
    positive
].astype(int)
source_gt_overlap = overlap_counts[
    positive
].astype(int)

order = np.argsort(
    -source_gt_overlap
)
source_gt_ids = source_gt_ids[order]
source_gt_overlap = source_gt_overlap[order]

target_ids_cpu = torch.as_tensor(
    target["ids"]
).detach().cpu().long()

id_to_target_row = {
    int(gt_id): row
    for row, gt_id
    in enumerate(target_ids_cpu.tolist())
}

if "centers_um" in target:
    all_gt_centers_um = torch.as_tensor(
        target["centers_um"]
    ).float()
else:
    all_gt_centers_um = (
        torch.as_tensor(
            target["centers_cellscale"]
        ).float()
        * float(
            batch["dref_um"][0]
        )
    )

source_gt_rows = torch.tensor(
    [
        id_to_target_row[int(gt_id)]
        for gt_id in source_gt_ids
    ],
    dtype=torch.long,
)

source_gt_centers_um = (
    all_gt_centers_um[
        source_gt_rows
    ]
    .detach()
    .cpu()
    .float()
)

print("Scene:", sample)
print("GT cells overlapping source 9:", len(source_gt_ids))
display(pd.DataFrame({
    "gt_id": source_gt_ids,
    "overlap_voxels_with_source9": source_gt_overlap,
    "center_z_um": source_gt_centers_um[:, 0].numpy(),
    "center_y_um": source_gt_centers_um[:, 1].numpy(),
    "center_x_um": source_gt_centers_um[:, 2].numpy(),
}))

if len(source_gt_ids) < 2:
    raise RuntimeError(
        "Source 9 does not overlap multiple GT cells; this is not a merge diagnostic."
    )

## 2. Build a **pure-spatial cache** once

The final trained weights are used, but temporal co-reasoning is bypassed:

```text
spatial encoder
→ spatial decoder E3→E2→D1→D0
→ dense heads
```

No GNN, no CR1, no CR2, no temporal memory.

In [ ]:
@torch.no_grad()
def build_pure_spatial_cache(model, b):
    model.eval()

    with torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
    ):
        acquisition = model.acquisition(
            b["spacing_um"],
            b["dref_um"],
        )

        pyramid = model.encoder(
            b["spatial_inputs"],
            b["spacing_um"],
            acquisition,
            b.get("spatial_padding_mask"),
        )

        # Pure spatial path: no CR1.
        e3 = pyramid.features[3]

        e2 = model.decoder.decode_to_e2(
            e3,
            pyramid,
            acquisition,
        )

        # Pure spatial path: no CR2.
        d1, d0, mask_features = (
            model.decoder.decode_from_e2(
                e2,
                pyramid,
                acquisition,
            )
        )

        dense = model.dense_heads(d0)

    return {
        "pyramid": pyramid,
        "e3": e3.detach(),
        "e2": e2.detach(),
        "d1": d1.detach(),
        "d0": d0.detach(),
        "mask_features": mask_features.detach(),
        "dense": {
            key: value.detach()
            for key, value
            in dense.items()
        },
        "spacings": [
            pyramid.spacings_um[3].detach(),
            pyramid.spacings_um[2].detach(),
            pyramid.spacings_um[1].detach(),
            pyramid.spacings_um[0].detach(),
        ],
    }

model = StirNet(cfg).to(device)

checkpoint_info = load_checkpoint(
    FINAL_CHECKPOINT,
    model,
    optimizer=None,
    scheduler=None,
    scaler=None,
    map_location="cpu",
    strict=True,
    migrate_history=True,
)
model.eval()

torch.cuda.reset_peak_memory_stats()
cache_start = time.perf_counter()

SPATIAL = build_pure_spatial_cache(
    model,
    b,
)

print(
    f"Pure spatial cache built in {time.perf_counter()-cache_start:.2f}s"
)
print(
    "E3/E2/D1/D0:",
    tuple(SPATIAL["e3"].shape),
    tuple(SPATIAL["e2"].shape),
    tuple(SPATIAL["d1"].shape),
    tuple(SPATIAL["d0"].shape),
)
print(
    "Peak CUDA:",
    round(
        torch.cuda.max_memory_allocated()
        / 1024**3,
        3,
    ),
    "GiB",
)

# Gate A — does the dense spatial pathway already see the separations?

Two direct signals matter:

1. **internal boundary evidence** between GT cells that currently belong to the same CC;
2. **center heatmap peaks** corresponding to the individual GT cells.

If these signals are already strong, the CNN/spatial decoder contains the information and the failure is later in query decomposition.

## 3. Build an internal GT-boundary mask for source 9

Only boundaries **between two different nonzero GT instances** are counted. Outer foreground/background boundary is excluded.

In [ ]:
def internal_instance_boundary(labels):
    labels = np.asarray(labels)
    boundary = np.zeros_like(
        labels,
        dtype=bool,
    )

    for axis in range(3):
        # Forward neighbour.
        a = [slice(None)] * 3
        c = [slice(None)] * 3
        a[axis] = slice(0, -1)
        c[axis] = slice(1, None)

        la = labels[tuple(a)]
        lc = labels[tuple(c)]

        diff = (
            (la > 0)
            & (lc > 0)
            & (la != lc)
        )

        ba = boundary[tuple(a)]
        bc = boundary[tuple(c)]
        ba |= diff
        bc |= diff

    return boundary

gt_internal_boundary_cpu = (
    internal_instance_boundary(
        gt_label_map_cpu
    )
)

source_internal_boundary_cpu = (
    gt_internal_boundary_cpu
    & source_mask_cpu
)

print(
    "Internal GT-boundary voxels inside source 9:",
    int(
        source_internal_boundary_cpu.sum()
    ),
)

## 4. Evaluate the learned dense boundary head inside source 9

In [ ]:
def binary_auc(scores, labels):
    scores = np.asarray(
        scores,
        dtype=np.float64,
    )
    labels = np.asarray(
        labels,
        dtype=np.int64,
    )
    pos = labels == 1
    neg = labels == 0

    if pos.sum() == 0 or neg.sum() == 0:
        return float("nan")

    ranks = rankdata(scores)
    n_pos = int(pos.sum())
    n_neg = int(neg.sum())

    return float(
        (
            ranks[pos].sum()
            - n_pos * (n_pos + 1) / 2
        )
        / (n_pos * n_neg)
    )

boundary_prob = (
    SPATIAL["dense"][
        "boundary_logits"
    ][0, 0]
    .sigmoid()
    .float()
)

d0_shape = tuple(
    int(v)
    for v
    in boundary_prob.shape
)

source_mask_d0 = resize_label_map_nearest(
    torch.from_numpy(
        source_mask_cpu.astype(
            np.int32
        )
    ).to(device),
    d0_shape,
).bool()

internal_boundary_d0 = resize_label_map_nearest(
    torch.from_numpy(
        source_internal_boundary_cpu.astype(
            np.int32
        )
    ).to(device),
    d0_shape,
).bool()

positive_scores = (
    boundary_prob[
        internal_boundary_d0
    ]
    .detach()
    .cpu()
    .numpy()
)

negative_mask = (
    source_mask_d0
    & ~internal_boundary_d0
)

negative_scores = (
    boundary_prob[
        negative_mask
    ]
    .detach()
    .cpu()
    .numpy()
)

scores = np.concatenate(
    [
        positive_scores,
        negative_scores,
    ]
)
labels = np.concatenate(
    [
        np.ones(
            len(positive_scores),
            dtype=np.int64,
        ),
        np.zeros(
            len(negative_scores),
            dtype=np.int64,
        ),
    ]
)

boundary_stats = {
    "internal_boundary_auc": binary_auc(
        scores,
        labels,
    ),
    "internal_boundary_mean_probability": float(
        np.mean(positive_scores)
    ),
    "source9_nonboundary_mean_probability": float(
        np.mean(negative_scores)
    ),
    "internal_boundary_recall_at_0p5": float(
        np.mean(
            positive_scores >= 0.5
        )
    ),
    "positive_voxels": int(
        len(positive_scores)
    ),
    "negative_voxels": int(
        len(negative_scores)
    ),
}

display(
    pd.DataFrame(
        [boundary_stats]
    )
)

## 5. Extract learned center-heatmap peaks inside source 9

This is deliberately simple and diagnostic:

- local maxima from the dense center heatmap;
- greedy physical NMS;
- keep enough candidates to cover the number of GT cells;
- match predicted peaks to GT centers by physical distance.

No GT center is used to generate the peaks.

In [ ]:
center_prob = (
    SPATIAL["dense"][
        "center_heatmap_logits"
    ][0, 0]
    .sigmoid()
    .float()
)

center_shape = tuple(
    int(v)
    for v
    in center_prob.shape
)

source_mask_center = resize_label_map_nearest(
    torch.from_numpy(
        source_mask_cpu.astype(
            np.int32
        )
    ).to(device),
    center_shape,
).bool()

spacing_d0 = (
    SPATIAL["spacings"][3][0]
    .detach()
    .float()
)

coords_d0 = (
    feature_grid_coordinates_um(
        center_shape,
        SPATIAL["spacings"][3],
        relative_to_center=True,
    )[0]
    .float()
)

with torch.no_grad():
    pooled = F.max_pool3d(
        center_prob[
            None, None
        ],
        kernel_size=3,
        stride=1,
        padding=1,
    )[0, 0]

    local_max = (
        center_prob >= pooled
    ) & source_mask_center

    candidate_index = torch.nonzero(
        local_max.flatten(),
        as_tuple=False,
    ).flatten()

    candidate_score = (
        center_prob.flatten()[
            candidate_index
        ]
    )

    sort_order = torch.argsort(
        candidate_score,
        descending=True,
    )

    candidate_index = (
        candidate_index[sort_order]
    )
    candidate_score = (
        candidate_score[sort_order]
    )

# Physical NMS.
dref = float(
    b["dref_um"][0]
    .detach()
    .cpu()
)

min_peak_distance_um = (
    0.45 * dref
)

selected_index = []
selected_scores = []
selected_coords = []

for idx, score in zip(
    candidate_index.tolist(),
    candidate_score.tolist(),
):
    coord = (
        coords_d0[idx]
        .detach()
        .cpu()
    )

    if selected_coords:
        distances = torch.linalg.vector_norm(
            torch.stack(
                selected_coords
            ) - coord[None],
            dim=-1,
        )
        if float(
            distances.min()
        ) < min_peak_distance_um:
            continue

    selected_index.append(
        int(idx)
    )
    selected_scores.append(
        float(score)
    )
    selected_coords.append(
        coord
    )

    if len(selected_coords) >= max(
        2 * len(source_gt_ids),
        len(source_gt_ids) + 4,
    ):
        break

selected_coords_um = (
    torch.stack(selected_coords)
    if selected_coords
    else torch.zeros(
        (0, 3),
        dtype=torch.float32,
    )
)

# Match the strongest K spatial peaks to GT centers.
K = len(source_gt_ids)
peak_coords_for_match = (
    selected_coords_um[:K]
)

if len(peak_coords_for_match):
    distance_matrix = torch.cdist(
        peak_coords_for_match.float(),
        source_gt_centers_um.float(),
    ).numpy()

    row_ind, col_ind = (
        linear_sum_assignment(
            distance_matrix
        )
    )
    matched_distances = (
        distance_matrix[
            row_ind,
            col_ind,
        ]
    )
else:
    row_ind = np.zeros(
        0,
        dtype=int,
    )
    col_ind = np.zeros(
        0,
        dtype=int,
    )
    matched_distances = np.zeros(
        0,
        dtype=float,
    )

peak_stats = {
    "gt_cells": K,
    "nms_peak_candidates": int(
        len(selected_coords_um)
    ),
    "topK_available": int(
        len(peak_coords_for_match)
    ),
    "matched_peak_mean_distance_um": (
        float(
            np.mean(
                matched_distances
            )
        )
        if len(matched_distances)
        else float("nan")
    ),
    "matched_peak_median_distance_um": (
        float(
            np.median(
                matched_distances
            )
        )
        if len(matched_distances)
        else float("nan")
    ),
    "matched_within_0p5_dref": (
        int(
            np.sum(
                matched_distances
                <= 0.5 * dref
            )
        )
        if len(matched_distances)
        else 0
    ),
}

display(
    pd.DataFrame(
        [peak_stats]
    )
)

peak_table = pd.DataFrame({
    "peak_rank": np.arange(
        len(selected_coords_um)
    ),
    "score": selected_scores,
    "z_um": (
        selected_coords_um[:, 0].numpy()
        if len(selected_coords_um)
        else []
    ),
    "y_um": (
        selected_coords_um[:, 1].numpy()
        if len(selected_coords_um)
        else []
    ),
    "x_um": (
        selected_coords_um[:, 2].numpy()
        if len(selected_coords_um)
        else []
    ),
})

display(
    peak_table.head(
        max(
            K + 4,
            12,
        )
    )
)

# Gate B — do E2/D1 spatial features distinguish the individual GT cells?

This is supportive evidence, not the final decision.

A good spatial segmentation network does not necessarily encode a globally unique appearance vector for each cell. Still, if E2/D1 features are completely homogeneous across all nine cells, that is evidence against the spatial representation.

In [ ]:
def feature_separability(
    feature,
    gt_ids,
    *,
    name,
):
    feature = (
        feature[0]
        .detach()
        .float()
    )

    shape = tuple(
        int(v)
        for v in feature.shape[-3:]
    )

    gt_ds = resize_label_map_nearest(
        torch.from_numpy(
            gt_label_map_cpu.astype(
                np.int32
            )
        ).to(device),
        shape,
    )

    vectors = []
    labels = []

    centroids = []

    for class_index, gt_id in enumerate(
        gt_ids
    ):
        mask = (
            gt_ds == int(gt_id)
        )

        voxels = (
            feature
            .permute(
                1, 2, 3, 0
            )[mask]
        )

        if len(voxels) == 0:
            continue

        # Bound diagnostic memory.
        if len(voxels) > 256:
            indices = torch.linspace(
                0,
                len(voxels) - 1,
                256,
                device=voxels.device,
            ).long()
            voxels = voxels[
                indices
            ]

        center = voxels.mean(
            dim=0
        )
        centroids.append(
            center
        )

        vectors.append(
            voxels
        )
        labels.append(
            torch.full(
                (len(voxels),),
                class_index,
                device=voxels.device,
                dtype=torch.long,
            )
        )

    if len(centroids) < 2:
        return {
            "feature": name,
            "classes": len(centroids),
            "centroid_cos_mean": float("nan"),
            "centroid_pair_l2_mean": float("nan"),
            "nearest_centroid_accuracy": float("nan"),
        }

    centroids = torch.stack(
        centroids
    )
    x = torch.cat(
        vectors,
        dim=0,
    )
    y = torch.cat(
        labels,
        dim=0,
    )

    cosine = (
        F.normalize(
            centroids,
            dim=-1,
        )
        @ F.normalize(
            centroids,
            dim=-1,
        ).T
    )

    mask = ~torch.eye(
        len(centroids),
        device=centroids.device,
        dtype=torch.bool,
    )
    cosine_values = cosine[
        mask
    ]

    pair_l2 = torch.pdist(
        centroids
    )

    # Simple nearest centroid classification of feature-grid voxels.
    distance = torch.cdist(
        x,
        centroids,
    )
    prediction = distance.argmin(
        dim=-1
    )

    accuracy = (
        prediction == y
    ).float().mean()

    return {
        "feature": name,
        "classes": len(centroids),
        "centroid_cos_mean": float(
            cosine_values.mean()
            .detach()
            .cpu()
        ),
        "centroid_cos_median": float(
            cosine_values.median()
            .detach()
            .cpu()
        ),
        "centroid_pair_l2_mean": float(
            pair_l2.mean()
            .detach()
            .cpu()
        ),
        "nearest_centroid_accuracy": float(
            accuracy.detach().cpu()
        ),
        "chance_accuracy": (
            1.0
            / len(centroids)
        ),
    }

feature_df = pd.DataFrame([
    feature_separability(
        SPATIAL["e2"],
        source_gt_ids,
        name="E2",
    ),
    feature_separability(
        SPATIAL["d1"],
        source_gt_ids,
        name="D1",
    ),
])

display(feature_df)

# Gate C — pure-spatial query counterfactuals

Now we remove temporal memory completely.

This is the most important part of Notebook 19.

The model is asked:

> If I give the spatial query decoder better spatial references, can it actually produce different cell masks?

## 6. Construct a valid empty temporal state

This prevents QueryBuilder from creating temporal queries and prevents any temporal memory from entering the decoder.

In [ ]:
def empty_temporal_state(model):
    d = int(
        model.cfg.temporal.d_model
    )
    edge_dim = int(
        model.cfg.temporal.hypothesis_edge_dim
    )

    return TemporalState(
        tokens=torch.zeros(
            (0, d),
            device=device,
            dtype=SPATIAL["e2"].dtype,
        ),
        ref_um=torch.zeros(
            (0, 3),
            device=device,
            dtype=torch.float32,
        ),
        ref_cellscale=torch.zeros(
            (0, 3),
            device=device,
            dtype=torch.float32,
        ),
        salience=torch.zeros(
            (0, 1),
            device=device,
            dtype=SPATIAL["e2"].dtype,
        ),
        reliability=torch.zeros(
            (0, 1),
            device=device,
            dtype=SPATIAL["e2"].dtype,
        ),
        status=torch.zeros(
            (0, 10),
            device=device,
            dtype=torch.float32,
        ),
        edge_index=torch.zeros(
            (2, 0),
            device=device,
            dtype=torch.long,
        ),
        edge_attr=torch.zeros(
            (0, edge_dim),
            device=device,
            dtype=torch.float32,
        ),
        batch_index=torch.zeros(
            (0,),
            device=device,
            dtype=torch.long,
        ),
    )

EMPTY_TEMPORAL = empty_temporal_state(
    model
)

# Hard-disable component/query temporal memory for this notebook.
model.query_builder.component_memory_enabled = False
for layer in model.query_decoder.layers:
    layer.query_memory_enabled = False

## 7. Build the pure-spatial QueryState

In [ ]:
@torch.no_grad()
def build_spatial_qstate(model):
    model.eval()

    with torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
    ):
        qstate = model.query_builder(
            SPATIAL["e2"],
            SPATIAL["spacings"][1],
            b["instance_labels"],
            b["instance_features"],
            b["instance_ids"],
            b["instance_batch"],
            b["instance_centroids_um"],
            b["dref_um"],
            EMPTY_TEMPORAL,
            memory_ablation="full",
            return_debug=False,
            full_attention=False,
        )

    return qstate

BASE_QSTATE = build_spatial_qstate(
    model
)

def source9_query_indices(qstate):
    qtypes = qstate.query_types[0]
    source_ids = (
        qstate.source_instance_ids[0]
    )

    mask = (
        (source_ids == SOURCE_ID)
        & (
            (qtypes == QUERY_PRIMARY)
            | (qtypes == QUERY_SPLIT)
        )
    )

    return torch.nonzero(
        mask,
        as_tuple=False,
    ).flatten()

SOURCE9_Q = source9_query_indices(
    BASE_QSTATE
)

print(
    "Pure spatial query count:",
    int(
        (~BASE_QSTATE.padding_mask[0])
        .sum()
    ),
)
print(
    "Source-9 seeded queries:",
    SOURCE9_Q.tolist(),
)
print(
    "Source-9 seeded count:",
    len(SOURCE9_Q),
)
print(
    "GT cells in source 9:",
    len(source_gt_ids),
)

## 8. Helpers for normal vs split-local support

`split-local support` removes the **entire source-component mask** from split-query support. Split queries must then use their reference-local support and previous-mask support.

Primary queries keep the normal source support.

This is diagnostic only.

In [ ]:
ORIGINAL_SOURCE_SUPPORT = (
    model.query_decoder
    ._source_instance_support
)

def install_split_local_support(
    model,
    enabled,
):
    if not enabled:
        model.query_decoder._source_instance_support = (
            ORIGINAL_SOURCE_SUPPORT
        )
        return

    def split_local_support(
        q,
        instance_labels,
        spatial_shape,
    ):
        support = ORIGINAL_SOURCE_SUPPORT(
            q,
            instance_labels,
            spatial_shape,
        )

        split = (
            q.query_types
            == QUERY_SPLIT
        )

        support = support.clone()
        support[
            split[
                ..., None, None, None
            ].expand_as(
                support
            )
        ] = False

        return support

    model.query_decoder._source_instance_support = (
        split_local_support
    )

## 9. Run the spatial decoder and score source-9 masks against source-9 GT

Matching here is a local diagnostic Hungarian assignment based on soft Dice among only the source-9 seeded queries and the GT cells contained in source 9.

In [ ]:
@torch.no_grad()
def decode_spatial_qstate(
    model,
    qstate,
    *,
    split_local_support=False,
):
    install_split_local_support(
        model,
        split_local_support,
    )

    model.eval()

    with torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
    ):
        final_q, outputs = (
            model.query_decoder(
                qstate,
                [
                    SPATIAL["e3"],
                    SPATIAL["e2"],
                    SPATIAL["d1"],
                ],
                [
                    SPATIAL["spacings"][0],
                    SPATIAL["spacings"][1],
                    SPATIAL["spacings"][2],
                ],
                b["instance_labels"],
                b["dref_um"],
                temporal=None,
            )
        )

    install_split_local_support(
        model,
        False,
    )

    return final_q, outputs


def source9_soft_dice_matrix(
    output_logits,
    q_indices,
):
    shape = tuple(
        int(v)
        for v
        in output_logits.shape[-3:]
    )

    gt_ds = resize_label_map_nearest(
        torch.from_numpy(
            gt_label_map_cpu.astype(
                np.int32
            )
        ).to(device),
        shape,
    )

    pred = (
        output_logits[
            0,
            q_indices,
        ]
        .float()
        .sigmoid()
        .flatten(1)
    )

    targets = torch.stack([
        (
            gt_ds
            == int(gt_id)
        ).float()
        for gt_id in source_gt_ids
    ]).flatten(1)

    intersection = (
        pred[:, None, :]
        * targets[None, :, :]
    ).sum(dim=-1)

    dice = (
        2 * intersection
        + 1e-6
    ) / (
        pred.sum(
            dim=-1
        )[:, None]
        + targets.sum(
            dim=-1
        )[None, :]
        + 1e-6
    )

    return dice


def query_variant_metrics(
    model,
    qstate,
    *,
    name,
    split_local_support=False,
):
    final_q, outputs = (
        decode_spatial_qstate(
            model,
            qstate,
            split_local_support=split_local_support,
        )
    )

    final = outputs[-1]
    q_indices = source9_query_indices(
        final_q
    )

    dice = source9_soft_dice_matrix(
        final["coarse_mask_logits"],
        q_indices,
    )

    cost = (
        -dice.detach()
        .cpu()
        .numpy()
    )
    row_ind, col_ind = (
        linear_sum_assignment(
            cost
        )
    )

    assigned = dice[
        row_ind,
        col_ind,
    ]

    masks = (
        final["coarse_mask_logits"][
            0,
            q_indices,
        ]
        .float()
        .sigmoid()
        .flatten(1)
    )

    pair_intersection = (
        masks
        @ masks.T
    )
    sums = masks.sum(
        dim=-1
    )
    pair_dice = (
        2 * pair_intersection
        + 1e-6
    ) / (
        sums[:, None]
        + sums[None, :]
        + 1e-6
    )

    off = ~torch.eye(
        len(q_indices),
        device=device,
        dtype=torch.bool,
    )

    centers_um = (
        final_q.references_cellscale[
            0,
            q_indices,
        ]
        .float()
        * b["dref_um"][0].float()
    )

    center_distance = torch.cdist(
        centers_um,
        source_gt_centers_um.to(
            device
        ),
    )
    cr, cg = linear_sum_assignment(
        center_distance.detach()
        .cpu()
        .numpy()
    )

    return {
        "name": name,
        "split_local_support": split_local_support,
        "source9_queries": int(
            len(q_indices)
        ),
        "best_match_soft_dice_mean": float(
            assigned.mean()
            .detach()
            .cpu()
        ),
        "best_match_soft_dice_median": float(
            assigned.median()
            .detach()
            .cpu()
        ),
        "sibling_pair_mask_dice_mean": float(
            pair_dice[
                off
            ].mean()
            .detach()
            .cpu()
        ),
        "center_to_gt_mean_um": float(
            center_distance[
                cr,
                cg,
            ].mean()
            .detach()
            .cpu()
        ),
        "center_to_gt_median_um": float(
            center_distance[
                cr,
                cg,
            ].median()
            .detach()
            .cpu()
        ),
        "matched_cells": int(
            len(assigned)
        ),
    }, final_q, outputs

## 10. Build baseline, oracle-center, and learned-spatial-peak query states

In [ ]:
def with_source9_references(
    qstate,
    refs_um,
):
    refs = (
        qstate.references_cellscale
        .clone()
    )

    q_indices = source9_query_indices(
        qstate
    )

    count = min(
        len(q_indices),
        len(refs_um),
    )

    refs[
        0,
        q_indices[:count],
    ] = (
        refs_um[:count]
        .to(
            refs.device,
            dtype=refs.dtype,
        )
        / b["dref_um"][0]
        .to(
            refs.device,
            dtype=refs.dtype,
        )
    )

    return replace(
        qstate,
        references_cellscale=refs,
    )


# Deterministic GT-center order for the oracle diagnostic.
gt_sort = torch.argsort(
    source_gt_centers_um[:, 0]
    * 1e6
    + source_gt_centers_um[:, 1]
    * 1e3
    + source_gt_centers_um[:, 2]
)

oracle_centers_um = (
    source_gt_centers_um[
        gt_sort
    ]
)

ORACLE_QSTATE = (
    with_source9_references(
        BASE_QSTATE,
        oracle_centers_um,
    )
)

# Use the strongest physically separated learned center peaks.
peak_centers_um = (
    selected_coords_um[
        : len(SOURCE9_Q)
    ]
)

PEAK_QSTATE = (
    with_source9_references(
        BASE_QSTATE,
        peak_centers_um,
    )
)

print(
    "Oracle centers available:",
    len(oracle_centers_um),
)
print(
    "Learned spatial peaks available:",
    len(peak_centers_um),
)

## 11. Decisive forward-only spatial counterfactual table

In [ ]:
variant_results = []
variant_objects = {}

for name, qstate, local_support in [
    (
        "baseline_shared_centroid",
        BASE_QSTATE,
        False,
    ),
    (
        "oracle_centers_normal_support",
        ORACLE_QSTATE,
        False,
    ),
    (
        "spatial_peaks_normal_support",
        PEAK_QSTATE,
        False,
    ),
    (
        "oracle_centers_split_local_support",
        ORACLE_QSTATE,
        True,
    ),
    (
        "spatial_peaks_split_local_support",
        PEAK_QSTATE,
        True,
    ),
]:
    result, final_q, outputs = (
        query_variant_metrics(
            model,
            qstate,
            name=name,
            split_local_support=local_support,
        )
    )

    variant_results.append(
        result
    )
    variant_objects[name] = (
        final_q,
        outputs,
    )

variant_df = pd.DataFrame(
    variant_results
)

variant_df.to_csv(
    RUN_DIR
    / "spatial_counterfactuals.csv",
    index=False,
)

display(variant_df)

# Gate D — 5-step cached spatial-only query micro-overfit

This is still **not a full overfit**.

The CNN/spatial cache is fixed. No temporal path exists.

Two branches start from the same final checkpoint:

1. `normal_support`
2. `split_local_support`

Only QueryBuilder + QueryDecoder are updated for five query-bootstrap steps.

In [ ]:
QUERY_STAGE = curriculum_stage(
    cfg.curriculum,
    50,
)

def build_query_criterion():
    criterion = RefinementCriterion(
        cfg.losses,
        cfg.queries,
        cfg.training,
    ).to(device)
    criterion.set_loss_weight_overrides(
        QUERY_STAGE.loss_weight_overrides
    )
    return criterion


def query_objective(
    criterion,
    qstate,
    outputs,
    initial_refs,
):
    final_layer = outputs[-1]

    final = {
        "exist_logits": final_layer[
            "exist_logits"
        ],
        "centers_cellscale": final_layer[
            "centers_cellscale"
        ],
        "coarse_mask_logits": final_layer[
            "coarse_mask_logits"
        ],
        "coarse_spacing_um": final_layer[
            "coarse_spacing_um"
        ],
        "dref_um": b["dref_um"],
        "query_types": qstate.query_types,
        "source_instance_ids": (
            qstate.source_instance_ids
        ),
        "query_initial_references_cellscale": (
            initial_refs
        ),
    }

    coarse_targets = (
        criterion._coarse_targets(
            final,
            b["targets"],
        )
    )
    matches = criterion._match(
        final,
        qstate.padding_mask,
        b["targets"],
        coarse_targets,
    )

    weights = (
        criterion._effective_loss_weights()
    )

    loss_exist = (
        criterion._existence_loss(
            final["exist_logits"],
            qstate.padding_mask,
            matches,
        )
    )
    (
        loss_dice,
        loss_focal,
        loss_center,
    ) = criterion._coarse_losses(
        final,
        matches,
        b["targets"],
        coarse_targets,
    )

    total = (
        weights["exist"]
        * loss_exist
        + weights["dice_coarse"]
        * loss_dice
        + weights["focal_coarse"]
        * loss_focal
        + weights["center"]
        * loss_center
    )

    zero = total * 0
    aux_total = zero

    if weights["aux_layer"] > 0:
        for aux in outputs[:-1]:
            aux_for = {
                **aux,
                "dref_um": b["dref_um"],
            }

            aux_exist = (
                criterion._existence_loss(
                    aux["exist_logits"],
                    qstate.padding_mask,
                    matches,
                )
            )

            aux_targets = (
                criterion._coarse_targets(
                    aux_for,
                    b["targets"],
                )
            )

            (
                aux_dice,
                aux_focal,
                aux_center,
            ) = criterion._coarse_losses(
                aux_for,
                matches,
                b["targets"],
                aux_targets,
            )

            aux_total = (
                aux_total
                + weights["aux_layer"]
                * (
                    weights["exist"]
                    * aux_exist
                    + weights["dice_coarse"]
                    * aux_dice
                    + weights["focal_coarse"]
                    * aux_focal
                    + weights["center"]
                    * aux_center
                )
            )

    total = total + aux_total

    return {
        "loss": total,
        "dice_coarse": loss_dice,
        "center": loss_center,
        "matches": matches,
    }

In [ ]:
def build_spatial_branch(
    *,
    split_local_support,
):
    branch = StirNet(
        cfg
    ).to(device)

    load_checkpoint(
        FINAL_CHECKPOINT,
        branch,
        optimizer=None,
        scheduler=None,
        scaler=None,
        map_location="cpu",
        strict=True,
        migrate_history=True,
    )

    branch.query_builder.component_memory_enabled = False
    for layer in branch.query_decoder.layers:
        layer.query_memory_enabled = False

    # Freeze everything except query modules.
    for parameter in branch.parameters():
        parameter.requires_grad_(False)

    for parameter in branch.query_builder.parameters():
        parameter.requires_grad_(True)

    for parameter in branch.query_decoder.parameters():
        parameter.requires_grad_(True)

    criterion = (
        build_query_criterion()
    )

    optimizer = torch.optim.AdamW(
        [
            p
            for p in branch.parameters()
            if p.requires_grad
        ],
        lr=float(
            cfg.training.lr
        ),
        weight_decay=float(
            cfg.training.weight_decay
        ),
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=True,
        init_scale=1024.0,
    )

    return (
        branch,
        criterion,
        optimizer,
        scaler,
        split_local_support,
    )


def cached_spatial_train_forward(
    branch,
    *,
    split_local_support,
):
    qstate = branch.query_builder(
        SPATIAL["e2"],
        SPATIAL["spacings"][1],
        b["instance_labels"],
        b["instance_features"],
        b["instance_ids"],
        b["instance_batch"],
        b["instance_centroids_um"],
        b["dref_um"],
        EMPTY_TEMPORAL,
    )

    initial_refs = (
        qstate.references_cellscale
    )

    # The normal-support function was bound from the main decoder object,
    # so build a branch-local equivalent rather than sharing a bound method.
    if split_local_support:
        original = (
            branch.query_decoder
            ._source_instance_support
        )

        def branch_local_support(
            q,
            instance_labels,
            spatial_shape,
        ):
            support = original(
                q,
                instance_labels,
                spatial_shape,
            )
            split = (
                q.query_types
                == QUERY_SPLIT
            )
            support = support.clone()
            support[
                split[
                    ..., None, None, None
                ].expand_as(
                    support
                )
            ] = False
            return support

        branch.query_decoder._source_instance_support = (
            branch_local_support
        )

    qstate, outputs = (
        branch.query_decoder(
            qstate,
            [
                SPATIAL["e3"],
                SPATIAL["e2"],
                SPATIAL["d1"],
            ],
            [
                SPATIAL["spacings"][0],
                SPATIAL["spacings"][1],
                SPATIAL["spacings"][2],
            ],
            b["instance_labels"],
            b["dref_um"],
            temporal=None,
        )
    )

    return (
        qstate,
        outputs,
        initial_refs,
    )

### Important implementation note

A fresh branch is reconstructed for each support experiment so monkey-patching cannot leak between branches.

In [ ]:
def micro_branch(
    name,
    *,
    split_local_support,
):
    (
        branch,
        criterion,
        optimizer,
        scaler,
        local_support,
    ) = build_spatial_branch(
        split_local_support=split_local_support
    )

    history = []

    for step in range(
        MICRO_STEPS + 1
    ):
        if step in EVAL_STEPS:
            branch.eval()
            with torch.no_grad(), torch.autocast(
                device_type="cuda",
                dtype=AMP_DTYPE,
            ):
                qstate, outputs, initial_refs = (
                    cached_spatial_train_forward(
                        branch,
                        split_local_support=local_support,
                    )
                )

                objective = (
                    query_objective(
                        criterion,
                        qstate,
                        outputs,
                        initial_refs,
                    )
                )

            # Local source-9 mask quality.
            q_indices = source9_query_indices(
                qstate
            )
            dice = source9_soft_dice_matrix(
                outputs[-1][
                    "coarse_mask_logits"
                ],
                q_indices,
            )
            r, c = linear_sum_assignment(
                -dice.detach()
                .cpu()
                .numpy()
            )
            assigned = dice[
                r,
                c,
            ]

            centers_um = (
                qstate.references_cellscale[
                    0,
                    q_indices,
                ]
                .float()
                * b["dref_um"][0]
                .float()
            )
            center_distance = torch.cdist(
                centers_um,
                source_gt_centers_um.to(
                    device
                ),
            )
            cr, cg = linear_sum_assignment(
                center_distance.detach()
                .cpu()
                .numpy()
            )

            history.append({
                "branch": name,
                "micro_step": step,
                "query_loss": float(
                    objective[
                        "loss"
                    ].detach().cpu()
                ),
                "global_coarse_dice_loss": float(
                    objective[
                        "dice_coarse"
                    ].detach().cpu()
                ),
                "source9_best_soft_dice": float(
                    assigned.mean()
                    .detach()
                    .cpu()
                ),
                "source9_center_to_gt_mean_um": float(
                    center_distance[
                        cr,
                        cg,
                    ].mean()
                    .detach()
                    .cpu()
                ),
            })

        if step == MICRO_STEPS:
            break

        branch.eval()
        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=AMP_DTYPE,
        ):
            qstate, outputs, initial_refs = (
                cached_spatial_train_forward(
                    branch,
                    split_local_support=local_support,
                )
            )

            objective = query_objective(
                criterion,
                qstate,
                outputs,
                initial_refs,
            )

        scaler.scale(
            objective["loss"]
        ).backward()
        scaler.unscale_(
            optimizer
        )

        torch.nn.utils.clip_grad_norm_(
            [
                p
                for p
                in branch.parameters()
                if p.requires_grad
            ],
            float(
                cfg.training.max_grad_norm
            ),
        )

        scaler.step(
            optimizer
        )
        scaler.update()

    return pd.DataFrame(
        history
    )


print("Running 5-step normal-support spatial branch...")
normal_micro = micro_branch(
    "normal_support",
    split_local_support=False,
)

print("Running 5-step split-local-support spatial branch...")
local_micro = micro_branch(
    "split_local_support",
    split_local_support=True,
)

micro_df = pd.concat(
    [
        normal_micro,
        local_micro,
    ],
    ignore_index=True,
)

micro_df.to_csv(
    RUN_DIR
    / "spatial_micro_overfit.csv",
    index=False,
)

display(micro_df)

# Automatic diagnostic summary

The rules below are intentionally descriptive rather than claiming a single architectural cause from one scene.

In [ ]:
baseline_row = variant_df[
    variant_df["name"]
    == "baseline_shared_centroid"
].iloc[0]

oracle_row = variant_df[
    variant_df["name"]
    == "oracle_centers_normal_support"
].iloc[0]

peak_row = variant_df[
    variant_df["name"]
    == "spatial_peaks_normal_support"
].iloc[0]

oracle_local_row = variant_df[
    variant_df["name"]
    == "oracle_centers_split_local_support"
].iloc[0]

peak_local_row = variant_df[
    variant_df["name"]
    == "spatial_peaks_split_local_support"
].iloc[0]

evidence = []

if (
    boundary_stats[
        "internal_boundary_auc"
    ] >= 0.75
):
    evidence.append(
        "Dense boundary evidence is reasonably discriminative inside the merged source; "
        "the spatial representation contains internal separation signal."
    )
else:
    evidence.append(
        "Dense internal-boundary evidence is weak; the spatial backbone/decoder itself "
        "may not represent the touching-cell boundaries well enough."
    )

if (
    peak_stats[
        "matched_within_0p5_dref"
    ]
    >= max(
        2,
        int(
            0.7
            * len(
                source_gt_ids
            )
        ),
    )
):
    evidence.append(
        "The learned dense center head recovers most source-9 cell centers from spatial evidence alone."
    )
else:
    evidence.append(
        "The learned dense center head does not recover most source-9 cell centers reliably."
    )

oracle_gain = (
    oracle_row[
        "best_match_soft_dice_mean"
    ]
    - baseline_row[
        "best_match_soft_dice_mean"
    ]
)

peak_gain = (
    peak_row[
        "best_match_soft_dice_mean"
    ]
    - baseline_row[
        "best_match_soft_dice_mean"
    ]
)

oracle_local_gain = (
    oracle_local_row[
        "best_match_soft_dice_mean"
    ]
    - oracle_row[
        "best_match_soft_dice_mean"
    ]
)

peak_local_gain = (
    peak_local_row[
        "best_match_soft_dice_mean"
    ]
    - peak_row[
        "best_match_soft_dice_mean"
    ]
)

if oracle_gain > 0.05:
    evidence.append(
        "Oracle centers substantially improve source-9 coarse masks: query/reference localization "
        "is a major bottleneck rather than lack of spatial image information."
    )
else:
    evidence.append(
        "Even oracle centers do not substantially improve source-9 masks: the bottleneck is downstream "
        "of center initialization (spatial attention/support/mask representation) or in the spatial features themselves."
    )

if oracle_local_gain > 0.03:
    evidence.append(
        "Removing the whole merged-source support from split queries helps strongly: shared component support "
        "is diluting split-specific spatial attention."
    )

if peak_gain > 0.03:
    evidence.append(
        "Learned spatial center peaks already improve decomposition without GT; the dense center head is a viable "
        "source of split-query initialization."
    )

if peak_local_gain > 0.03:
    evidence.append(
        "Spatial peaks plus split-local support outperform ordinary peak seeding; query support design is implicated."
    )

final_normal = normal_micro.sort_values(
    "micro_step"
).iloc[-1]

final_local = local_micro.sort_values(
    "micro_step"
).iloc[-1]

micro_support_gain = (
    final_local[
        "source9_best_soft_dice"
    ]
    - final_normal[
        "source9_best_soft_dice"
    ]
)

if micro_support_gain > 0.02:
    evidence.append(
        "The 5-step spatial micro-test learns faster with split-local support, reinforcing the support-dilution hypothesis."
    )
elif (
    max(
        final_normal[
            "source9_best_soft_dice"
        ],
        final_local[
            "source9_best_soft_dice"
        ],
    )
    <= baseline_row[
        "best_match_soft_dice_mean"
    ]
    + 0.01
):
    evidence.append(
        "Five targeted spatial-query steps do not improve source-9 decomposition; simple additional query training "
        "is not enough and a structural spatial-query bottleneck remains."
    )

summary = {
    "boundary": boundary_stats,
    "peaks": peak_stats,
    "oracle_gain_soft_dice": float(
        oracle_gain
    ),
    "peak_gain_soft_dice": float(
        peak_gain
    ),
    "oracle_split_local_gain_soft_dice": float(
        oracle_local_gain
    ),
    "peak_split_local_gain_soft_dice": float(
        peak_local_gain
    ),
    "micro_split_local_gain_soft_dice": float(
        micro_support_gain
    ),
    "evidence": evidence,
}

print("=" * 70)
print("NOTEBOOK 19 SPATIAL DIAGNOSIS")
print("=" * 70)

for index, item in enumerate(
    evidence,
    start=1,
):
    print(f"{index}. {item}")

with (
    RUN_DIR
    / "diagnostic_summary.json"
).open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        summary,
        handle,
        indent=2,
        default=float,
    )

## Compact visualization

In [ ]:
ax = variant_df.set_index(
    "name"
)[
    "best_match_soft_dice_mean"
].plot(
    kind="bar",
    figsize=(10, 4),
)
ax.set_ylabel(
    "source-9 best-match soft Dice"
)
ax.set_title(
    "Spatial-only source-9 counterfactuals"
)
plt.xticks(
    rotation=35,
    ha="right",
)
plt.tight_layout()
plt.show()

pivot = micro_df.pivot(
    index="micro_step",
    columns="branch",
    values="source9_best_soft_dice",
)
ax = pivot.plot(
    marker="o",
    figsize=(7, 3.5),
)
ax.set_ylabel(
    "source-9 best-match soft Dice"
)
ax.set_title(
    "5-step cached spatial query test"
)
ax.grid(
    True,
    alpha=0.25,
)
plt.tight_layout()
plt.show()

## Save diagnostic artifacts

In [ ]:
feature_df.to_csv(
    RUN_DIR
    / "feature_separability.csv",
    index=False,
)

peak_table.to_csv(
    RUN_DIR
    / "dense_center_peaks.csv",
    index=False,
)

pd.DataFrame(
    [boundary_stats]
).to_csv(
    RUN_DIR
    / "internal_boundary_stats.csv",
    index=False,
)

pd.DataFrame(
    [peak_stats]
).to_csv(
    RUN_DIR
    / "center_peak_stats.csv",
    index=False,
)

print("Artifacts:")
for path in sorted(
    RUN_DIR.glob("*")
):
    print(" ", path.name)

gc.collect()
torch.cuda.empty_cache()
print("CUDA cache cleared.")

# Decision guide

Read the forward counterfactuals **before** interpreting the 5-step micro-test.

### Case 1 — boundary + center signals good, oracle centers good

The CNN/spatial decoder contains the cell-separation information.

The problem is primarily:

```text
shared component query
→ identical split references
→ failure to discover different spatial locations
```

A strong next design would use learned dense spatial peaks / center proposals to seed split queries.

### Case 2 — oracle centers good only with split-local support

The spatial features are adequate, but every split query being allowed to attend over the **entire merged source component** is suppressing specialization.

The next change should target query support, not the CNN.

### Case 3 — dense center peaks good, peak-seeded queries good

This is the best spatial result.

The model already predicts where the cells are from the current frame; STIR-Net simply is not converting those center proposals into split-query references.

### Case 4 — boundary evidence good, but oracle centers still produce poor masks

The image features know where internal boundaries are, but the query/mask decoder is not turning those features into instances.

Focus next on spatial cross-attention / mask embedding / support, not temporal reasoning.

### Case 5 — boundary evidence weak and center peaks missing

Now there is direct evidence that the **spatial representation itself** is insufficient for these easy touching cells.

Only in this case should we move upstream toward the CNN/dense spatial training objective or spatial architecture.

### Case 6 — 5-step cached query training does nothing

Do not run a full overfit just to see whether more steps rescue it.

Use the forward counterfactuals to choose the structural change first.